# PiLENS v2 on Kaggle: Steps 2, 3 and 4 in one "Run All"

**Pehle ye settings:**
1. *Add Input*: `minmints/ufc-crime-full-dataset` (aur agar pehle se features ka dataset hai to wo bhi).
2. *Settings*: Accelerator **GPU T4 x2**, Internet **On**.
3. Phir **Run All**. Har section ke end par `OK` print hota hai; error aaye to notebook wahin ruk jati hai.

Kya hota hai:
- **Step 2**: official splits (dedupe, 15% val, leak check) aur X3D-S features (32 clips/video, dono GPU par), `valid: N/N`.
- **Step 3**: 14-class (linear, mil_topk, tconv_mil; 4 folds x 3 seeds) aur binary MIL (mlp, tconv; 5 seeds).
- **Step 4**: ONNX export + parity check.
- End mein `pilens_outputs.zip` (splits, summaries, heads, ONNX, results tables) *Output* tab mein.

Poora run lagbhag kuch ghante ka hai (zyada waqt feature extraction ka). Kaggle ka 12 ghante ka session limit hai;
features ek baar ban jayein to *Output -> New Dataset* bana lo, agli dafa extraction skip ho jayegi.

In [ ]:
# ---- Settings (zaroorat ho to badlo) ----
REPO = "https://github.com/Huzzi-10/PiLENS-Real-Time-AI-Based-Suspicious-Activity-Detection-with-NightVision-on-Raspberry-Pi"
BRANCH = "main"
QUICK_TEST_LIMIT = 8          # pehle sirf 8 videos par extraction test
RUN_EXTRACTION = True         # False = sirf attached features dataset use karo
RUN_TRAINING = True
RUN_EXPORT = True
CLS14_HEADS = ["linear", "mil_topk", "tconv_mil"]
CLS14_SEEDS = [0, 1, 2]
MIL_HEADS = ["mlp", "tconv"]
MIL_SEEDS = [0, 1, 2, 3, 4]
DEPLOY_MIL = "mlp"            # Pi par jane wala binary head
DEPLOY_CLS = "linear"         # Pi par jane wala 14-class head (sabse sasta)
WORK = "/kaggle/working"

In [ ]:
import glob, os, subprocess, sys, time, json
from pathlib import Path

def sh(cmd, cwd=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

os.chdir(WORK)
if not Path("pilens").exists():
    sh(f"git clone -q -b {BRANCH} {REPO} pilens")
else:
    sh("git pull -q", cwd="pilens")
os.chdir(f"{WORK}/pilens")
sys.path.insert(0, f"{WORK}/pilens")  # so the notebook itself can import pilens_v2
sh("pip install -q pytorchvideo")
sh("git log --oneline -1")

# dataset root = the input that contains Anomaly_Train.txt
ROOT = None
for d in sorted(glob.glob("/kaggle/input/*")):
    if list(Path(d).rglob("Anomaly_Train.txt")):
        ROOT = d
        break
assert ROOT, "UCF-Crime dataset nahi mila: Add Input se minmints/ufc-crime-full-dataset add karo"
ANN = sorted(Path(ROOT).rglob("Temporal_Anomaly_Annotation*.txt"))
assert ANN, "Temporal_Anomaly_Annotation_for_Testing_Videos.txt nahi mili"
ANN = str(ANN[0])
import torch
print("ROOT =", ROOT, "\nANN  =", ANN, "\nGPUs =", torch.cuda.device_count())
print("OK")

## Step 2a: splits

In [ ]:
sh(f"python -m pilens_v2.data.splits --root {ROOT} --out splits --probe")
s = json.loads(Path("splits/split_summary.json").read_text())
a = s["tasks"]["anomaly"]
print(f"unique {s['unique_videos']}  duplicates removed {s['duplicates_removed']}  "
      f"anomaly train+val {a['train'] + a['val']}  test {a['test']}")
print("warnings:", s["warnings"] or "none")
for k, f in s["tasks"]["cls14"].items():
    print(k, "train", f["train"], "val", f["val"], "test", f["test"], "missing", len(f["missing_from_disk"]))
# expected (notes, section 4): 1900 unique, 50 duplicates, 1610 / 290, 168 test videos per fold
if (s["unique_videos"], s["duplicates_removed"], a["train"] + a["val"], a["test"]) != (1900, 50, 1610, 290):
    print("\nWARNING: counts notes se match nahi karte, upar ka output check karo")
print("OK")

## Step 2b: X3D-S features

In [ ]:
FEATS = None
# an attached features dataset with valid files for every video -> reuse it
from pilens_v2.train.extract_features import validate
import csv
ids = [r["video_id"] for r in csv.DictReader(open("splits/video_index.csv"))]
for d in sorted(glob.glob("/kaggle/input/*")):
    if Path(d, "video_meta.csv").exists():
        ok, bad = validate(d, ids)
        print(f"found features in {d}: valid {ok}/{len(ids)}")
        if ok == len(ids):
            FEATS = d
if FEATS is None and RUN_EXTRACTION:
    FEATS = f"{WORK}/feats_x3ds"
    base = f"python -m pilens_v2.train.extract_features --root {ROOT} --index splits/video_index.csv --out {FEATS}"
    sh(f"{base} --limit {QUICK_TEST_LIMIT}")                      # quick test
    n_gpu = max(1, torch.cuda.device_count())
    procs = [subprocess.Popen(f"{base} --shard {i}/{n_gpu} --device cuda:{i} > extract_{i}.log 2>&1", shell=True)
             for i in range(n_gpu)]
    last = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        if time.time() - last < 300:
            continue
        last = time.time()
        for i in range(n_gpu):
            tail = Path(f"extract_{i}.log").read_text().strip().splitlines()[-1:] if Path(f"extract_{i}.log").exists() else []
            print(f"[gpu {i}]", tail[0] if tail else "...", flush=True)
    assert all(p.returncode == 0 for p in procs), "extraction fail hui: extract_*.log dekho"
    sh(f"{base} --validate-only")
assert FEATS, "features nahi hain: RUN_EXTRACTION = True karo ya features dataset attach karo"
ok, bad = validate(FEATS, ids)
print(f"valid: {ok}/{len(ids)}")
assert ok == len(ids), bad[:10]
print("OK  ->  ab Output tab se feats_x3ds ka 'New Dataset' bana lo (backup)")

## Step 3: Task A (14-class) aur Task B (binary MIL)

In [ ]:
if RUN_TRAINING:
    seeds = " ".join(map(str, CLS14_SEEDS))
    for head in CLS14_HEADS:
        sh(f"python -m pilens_v2.train.train_cls14 --feats {FEATS} --splits splits --head {head} "
           f"--seeds {seeds} --out runs/cls14_{head}")
    seeds = " ".join(map(str, MIL_SEEDS))
    for head in MIL_HEADS:
        sh(f"python -m pilens_v2.train.train_mil --feats {FEATS} --splits splits --test-ann {ANN} "
           f"--head {head} --seeds {seeds} --out runs/mil_{head}")
print("OK")

## Step 4: ONNX export + parity

In [ ]:
if RUN_EXPORT:
    index = {r["video_id"]: r for r in csv.DictReader(open("splits/video_index.csv"))}
    test_ids = Path("splits/anomaly_test.txt").read_text().split()
    vid = next(v for v in test_ids if index[v]["label"] != "Normal")
    sh(f"python -m pilens_v2.export.export_onnx --out exports "
       f"--binary-ckpt runs/mil_{DEPLOY_MIL}/seed0.pt --cls-ckpt runs/cls14_{DEPLOY_CLS}/fold1_seed0.pt "
       f"--parity-video '{ROOT}/{index[vid]['rel_path']}'")
    print("threshold (validation):", json.loads(Path("exports/pilens_v2.json").read_text())["threshold"])
print("OK")

## Results tables + download

In [ ]:
sh("python -m pilens_v2.eval.make_tables --runs runs --out docs/v2_results.md")
import zipfile
with zipfile.ZipFile(f"{WORK}/pilens_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["splits/*", "runs/*/summary.json", "runs/*/*.pt", "exports/*", "docs/v2_results.md"]:
        for f in glob.glob(pattern):
            z.write(f)
print(f"OK -> {WORK}/pilens_outputs.zip (Output tab se download karo)")
print("Repo mein commit karne wali cheezein: splits/, runs/*/summary.json, docs/v2_results.md")
print("Pi par copy karne wali: exports/ (3 .onnx + pilens_v2.json)")